In [1]:
"""
============================================================
  Optimization Algorithms from Scratch
  Dataset: Synthetic Housing (16512 samples, 8 features)
  Algorithms:
    1. Gradient Descent (GD)
    2. Mini-Batch Gradient Descent (BGD)
    3. Momentum-Based GD
    4. AdaGrad
    5. NAG (Nesterov Accelerated Gradient)
    6. AdaDelta
    7. Adam
============================================================
"""

import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# ──────────────────────────────────────────────────────────
#  PLOT STYLING
# ──────────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.facecolor':  '#0d1117',
    'axes.facecolor':    '#161b22',
    'axes.edgecolor':    '#30363d',
    'text.color':        '#e6edf3',
    'axes.labelcolor':   '#e6edf3',
    'xtick.color':       '#8b949e',
    'ytick.color':       '#8b949e',
    'grid.color':        '#21262d',
    'grid.alpha':        0.8,
    'font.size':         11,
})

COLORS = {
    'GD':       '#ff6b6b',
    'BGD':      '#ffd93d',
    'Momentum': '#6bcb77',
    'AdaGrad':  '#4d96ff',
    'NAG':      '#c77dff',
    'AdaDelta': '#ff9f1c',
    'Adam':     '#2ec4b6',
}


# ══════════════════════════════════════════════════════════
#  SECTION 1 — DATASET
# ══════════════════════════════════════════════════════════

def make_dataset(n=16512, d=8, seed=42):
    """
    Generate a synthetic housing-like regression dataset.
    Returns preprocessed train/test sets with a bias column.
    """
    np.random.seed(seed)
    feature_names = ['MedInc','HouseAge','AveRooms','AveBedrms',
                     'Population','AveOccup','Latitude','Longitude']

    X = np.random.randn(n, d)
    true_w = np.array([0.8, 0.1, 0.05, -0.1, -0.05, -0.3, -0.4, -0.1])
    y = X @ true_w + 2.0 + 0.5 * np.random.randn(n)

    # Train/test split (80/20)
    idx   = np.random.permutation(n)
    split = int(0.8 * n)
    X_tr, X_te = X[idx[:split]], X[idx[split:]]
    y_tr, y_te = y[idx[:split]], y[idx[split:]]

    # Standardise
    mu_x, std_x = X_tr.mean(0), X_tr.std(0)
    mu_y, std_y = y_tr.mean(),  y_tr.std()

    X_tr_s = (X_tr - mu_x) / std_x
    X_te_s = (X_te - mu_x) / std_x
    y_tr_s = (y_tr - mu_y) / std_y
    y_te_s = (y_te - mu_y) / std_y

    # Add bias column
    X_tr_b = np.column_stack([np.ones(len(X_tr_s)), X_tr_s])
    X_te_b = np.column_stack([np.ones(len(X_te_s)), X_te_s])

    return X_tr_b, X_te_b, y_tr_s, y_te_s, X_tr, y_tr, feature_names


# ══════════════════════════════════════════════════════════
#  SECTION 2 — HELPER FUNCTIONS
# ══════════════════════════════════════════════════════════

def mse(X, y, w):
    """Mean Squared Error."""
    return np.mean((X @ w - y) ** 2)

def gradient(X, y, w):
    """Full-batch gradient of MSE."""
    n = len(y)
    return (2 / n) * X.T @ (X @ w - y)

def gradient_batch(Xb, yb, w):
    """Mini-batch gradient of MSE."""
    n = len(yb)
    return (2 / n) * Xb.T @ (Xb @ w - yb)


# ══════════════════════════════════════════════════════════
#  SECTION 3 — OPTIMIZERS
# ══════════════════════════════════════════════════════════

def gradient_descent(X_tr, y_tr, X_te, y_te, w0, lr=0.05, epochs=200):
    """
    1. Vanilla Gradient Descent (Full-Batch GD)
    Update: w = w - lr * ∇L(w)
    """
    w = w0.copy()
    train_loss, val_loss, grad_norm = [], [], []

    for _ in range(epochs):
        g = gradient(X_tr, y_tr, w)
        w -= lr * g

        train_loss.append(mse(X_tr, y_tr, w))
        val_loss.append(mse(X_te, y_te, w))
        grad_norm.append(np.linalg.norm(g))

    return w, train_loss, val_loss, grad_norm


def mini_batch_gd(X_tr, y_tr, X_te, y_te, w0, lr=0.05, batch_size=256, epochs=200):
    """
    2. Mini-Batch Gradient Descent
    Update: w = w - lr * ∇L_batch(w)
    Shuffles data each epoch and processes in chunks.
    """
    w = w0.copy()
    n = len(y_tr)
    train_loss, val_loss, grad_norm = [], [], []

    for _ in range(epochs):
        idx  = np.random.permutation(n)
        Xs, ys = X_tr[idx], y_tr[idx]
        epoch_gn, nb = 0, 0

        for start in range(0, n, batch_size):
            Xb = Xs[start:start + batch_size]
            yb = ys[start:start + batch_size]
            g  = gradient_batch(Xb, yb, w)
            w -= lr * g
            epoch_gn += np.linalg.norm(g)
            nb += 1

        train_loss.append(mse(X_tr, y_tr, w))
        val_loss.append(mse(X_te, y_te, w))
        grad_norm.append(epoch_gn / max(nb, 1))

    return w, train_loss, val_loss, grad_norm


def momentum_gd(X_tr, y_tr, X_te, y_te, w0, lr=0.05, beta=0.9, epochs=200):
    """
    3. Momentum-Based Gradient Descent
    v_t = beta * v_(t-1) + lr * ∇L(w)
    w   = w - v_t
    """
    w = w0.copy()
    v = np.zeros_like(w)
    train_loss, val_loss, grad_norm, velocity_norm = [], [], [], []

    for _ in range(epochs):
        g = gradient(X_tr, y_tr, w)
        v = beta * v + lr * g
        w -= v

        train_loss.append(mse(X_tr, y_tr, w))
        val_loss.append(mse(X_te, y_te, w))
        grad_norm.append(np.linalg.norm(g))
        velocity_norm.append(np.linalg.norm(v))

    return w, train_loss, val_loss, grad_norm, velocity_norm


def adagrad(X_tr, y_tr, X_te, y_te, w0, lr=0.5, eps=1e-8, epochs=200):
    """
    4. AdaGrad — Adaptive Gradient
    G_t = G_(t-1) + (∇L)^2
    w   = w - (lr / sqrt(G_t + eps)) * ∇L
    Learning rate shrinks per parameter based on history.
    """
    w = w0.copy()
    G = np.zeros_like(w)                        # accumulated squared gradients
    train_loss, val_loss, grad_norm, eff_lr = [], [], [], []

    for _ in range(epochs):
        g   = gradient(X_tr, y_tr, w)
        G  += g ** 2
        alr = lr / (np.sqrt(G) + eps)           # per-parameter learning rate
        w  -= alr * g

        train_loss.append(mse(X_tr, y_tr, w))
        val_loss.append(mse(X_te, y_te, w))
        grad_norm.append(np.linalg.norm(g))
        eff_lr.append(np.mean(alr))

    return w, train_loss, val_loss, grad_norm, eff_lr


def nag(X_tr, y_tr, X_te, y_te, w0, lr=0.05, beta=0.9, epochs=200):
    """
    5. NAG — Nesterov Accelerated Gradient
    w_lookahead = w - beta * v
    v_t = beta * v_(t-1) + lr * ∇L(w_lookahead)
    w   = w - v_t
    Computes gradient at the 'future' position (lookahead).
    """
    w = w0.copy()
    v = np.zeros_like(w)
    train_loss, val_loss, grad_norm, lookahead_dist = [], [], [], []

    for _ in range(epochs):
        w_look = w - beta * v                   # lookahead position
        g      = gradient(X_tr, y_tr, w_look)  # gradient at lookahead
        v      = beta * v + lr * g
        w     -= v

        train_loss.append(mse(X_tr, y_tr, w))
        val_loss.append(mse(X_te, y_te, w))
        grad_norm.append(np.linalg.norm(g))
        lookahead_dist.append(np.linalg.norm(w - w_look))

    return w, train_loss, val_loss, grad_norm, lookahead_dist


def adadelta(X_tr, y_tr, X_te, y_te, w0, rho=0.95, eps=1e-6, epochs=200):
    """
    6. AdaDelta — No global learning rate needed
    E[g²]_t  = rho * E[g²]_(t-1)  + (1-rho) * (∇L)²
    Δw       = -(sqrt(E[Δw²]_(t-1) + eps) / sqrt(E[g²]_t + eps)) * ∇L
    E[Δw²]_t = rho * E[Δw²]_(t-1) + (1-rho) * (Δw)²
    """
    w    = w0.copy()
    Eg2  = np.zeros_like(w)                     # running avg of squared gradients
    Edw2 = np.zeros_like(w)                     # running avg of squared updates
    train_loss, val_loss, grad_norm, update_norm = [], [], [], []

    for _ in range(epochs):
        g    = gradient(X_tr, y_tr, w)
        Eg2  = rho * Eg2  + (1 - rho) * g ** 2
        dw   = -(np.sqrt(Edw2 + eps) / np.sqrt(Eg2 + eps)) * g
        Edw2 = rho * Edw2 + (1 - rho) * dw ** 2
        w   += dw

        train_loss.append(mse(X_tr, y_tr, w))
        val_loss.append(mse(X_te, y_te, w))
        grad_norm.append(np.linalg.norm(g))
        update_norm.append(np.linalg.norm(dw))

    return w, train_loss, val_loss, grad_norm, update_norm


def adam(X_tr, y_tr, X_te, y_te, w0, lr=0.01, beta1=0.9, beta2=0.999, eps=1e-8, epochs=200):
    """
    7. Adam — Adaptive Moment Estimation
    m_t = beta1 * m_(t-1) + (1-beta1) * ∇L          ← 1st moment (mean)
    v_t = beta2 * v_(t-1) + (1-beta2) * (∇L)²        ← 2nd moment (variance)
    m̂  = m_t / (1 - beta1^t)                         ← bias correction
    v̂  = v_t / (1 - beta2^t)
    w   = w - lr * m̂ / (sqrt(v̂) + eps)
    """
    w  = w0.copy()
    m  = np.zeros_like(w)
    v  = np.zeros_like(w)
    train_loss, val_loss, grad_norm, m1_norm, m2_norm = [], [], [], [], []

    for t in range(1, epochs + 1):
        g = gradient(X_tr, y_tr, w)

        m = beta1 * m + (1 - beta1) * g
        v = beta2 * v + (1 - beta2) * g ** 2

        m_hat = m / (1 - beta1 ** t)           # bias-corrected 1st moment
        v_hat = v / (1 - beta2 ** t)           # bias-corrected 2nd moment

        w -= lr * m_hat / (np.sqrt(v_hat) + eps)

        train_loss.append(mse(X_tr, y_tr, w))
        val_loss.append(mse(X_te, y_te, w))
        grad_norm.append(np.linalg.norm(g))
        m1_norm.append(np.linalg.norm(m_hat))
        m2_norm.append(np.linalg.norm(v_hat))

    return w, train_loss, val_loss, grad_norm, m1_norm, m2_norm


# ══════════════════════════════════════════════════════════
#  SECTION 4 — PLOTTING FUNCTIONS
# ══════════════════════════════════════════════════════════

def save(fig, name):
    fig.savefig(name, dpi=130, bbox_inches='tight', facecolor='#0d1117')
    plt.close(fig)
    print(f'  ✅ Saved {name}')


def plot_dataset(X_tr, y_tr, feat_names):
    fig, axes = plt.subplots(1, 2, figsize=(16, 5))
    fig.suptitle('📊 Synthetic Housing Dataset  (16 512 samples · 8 features)',
                 fontsize=13, fontweight='bold', color='#e6edf3')

    # Target distribution
    axes[0].hist(y_tr, bins=50, color='#4d96ff', alpha=0.85, edgecolor='none')
    axes[0].axvline(y_tr.mean(), color='#ff6b6b', lw=2, ls='--',
                    label=f'Mean = {y_tr.mean():.2f}')
    axes[0].set_title('Target Distribution', fontweight='bold')
    axes[0].set_xlabel('House Value'); axes[0].set_ylabel('Frequency')
    axes[0].legend(); axes[0].grid(True, alpha=0.3)

    # Feature correlations
    cors  = [np.corrcoef(X_tr[:, i], y_tr)[0, 1] for i in range(X_tr.shape[1])]
    clrs  = ['#6bcb77' if c > 0 else '#ff6b6b' for c in cors]
    axes[1].barh(feat_names, cors, color=clrs, alpha=0.85)
    axes[1].axvline(0, color='#8b949e', lw=1)
    axes[1].set_title('Feature Correlations with Target', fontweight='bold')
    axes[1].set_xlabel('Pearson Correlation'); axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    save(fig, '00_dataset_overview.png')


def plot_3panel(title, fname, color, ep,
                train_l, val_l,
                p3, p3_ylabel, p3_title,
                extra_p0=None, extra_p3=None):
    """
    Generic 3-panel plot.
      Panel 0 : train vs val loss
      Panel 1 : log-scale loss
      Panel 2 : custom metric (p3) or extra_p3 lines
    extra_p0 / extra_p3 : list of (data, color, label) tuples for multi-line panels
    """
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle(title, fontsize=14, fontweight='bold', color='#e6edf3')

    # — Panel 0 : loss curves —
    if extra_p0:
        for data, c, lbl in extra_p0:
            axes[0].plot(data, color=c, lw=2, label=lbl)
        axes[0].legend()
    else:
        axes[0].plot(train_l, color=color, lw=2.5, label='Train')
        axes[0].plot(val_l,   color='#8b949e', lw=2, ls='--', label='Val')
        axes[0].fill_between(ep, train_l, alpha=0.12, color=color)
        axes[0].legend()
    axes[0].set_title('Train vs Validation Loss', fontweight='bold')
    axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('MSE Loss')
    axes[0].grid(True, alpha=0.4)

    # — Panel 1 : log scale —
    axes[1].semilogy(train_l, color=color,     lw=2.5, label='Train')
    axes[1].semilogy(val_l,   color='#8b949e', lw=2,   ls='--', label='Val')
    axes[1].set_title('Loss — Log Scale', fontweight='bold')
    axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('MSE (log)')
    axes[1].legend(); axes[1].grid(True, alpha=0.4)

    # — Panel 2 : custom metric —
    if extra_p3:
        for data, c, lbl in extra_p3:
            axes[2].plot(data, color=c, lw=2, label=lbl)
        axes[2].legend()
    else:
        axes[2].plot(p3, color='#c77dff', lw=2.5)
        axes[2].fill_between(ep, p3, alpha=0.12, color='#c77dff')
    axes[2].set_title(p3_title, fontweight='bold')
    axes[2].set_xlabel('Epoch'); axes[2].set_ylabel(p3_ylabel)
    axes[2].grid(True, alpha=0.4)

    plt.tight_layout()
    save(fig, fname)


def plot_all_comparison(ep, all_names, all_tr, all_vl, all_c):
    """Side-by-side comparison of all optimizers."""
    fig, axes = plt.subplots(1, 2, figsize=(20, 7))
    fig.suptitle('🏆 All Optimizers — Head-to-Head Comparison (200 Epochs)',
                 fontsize=16, fontweight='bold', color='#e6edf3')

    for n, tr, vl, c in zip(all_names, all_tr, all_vl, all_c):
        axes[0].plot(tr, color=c, lw=2.5, label=n)
        axes[1].plot(vl, color=c, lw=2.5, label=n)

    for ax, ttl in zip(axes, ['Training Loss', 'Validation Loss']):
        ax.set_title(ttl, fontweight='bold', fontsize=14)
        ax.set_xlabel('Epoch'); ax.set_ylabel('MSE Loss')
        ax.legend(fontsize=12); ax.grid(True, alpha=0.4)

    plt.tight_layout()
    save(fig, '08_all_comparison.png')


def plot_bar_summary(all_names, all_tr, all_vl, all_c):
    """Bar chart of final train / val losses."""
    ft = [t[-1] for t in all_tr]
    fv = [v[-1] for v in all_vl]

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    fig.suptitle('📊 Final Performance Summary — All Optimizers',
                 fontsize=15, fontweight='bold', color='#e6edf3')

    for ax, vals, ttl in zip(axes, [ft, fv],
                             ['Final Training Loss (↓ Better)',
                              'Final Validation Loss (↓ Better)']):
        bars = ax.bar(all_names, vals, color=all_c, alpha=0.85, edgecolor='none')
        ax.set_title(ttl, fontweight='bold')
        ax.set_ylabel('MSE Loss')
        ax.grid(True, alpha=0.4, axis='y')
        for bar, v in zip(bars, vals):
            ax.text(bar.get_x() + bar.get_width() / 2,
                    bar.get_height() + max(vals) * 0.01,
                    f'{v:.4f}', ha='center', va='bottom',
                    fontsize=9, color='#e6edf3')

    plt.tight_layout()
    save(fig, '09_final_summary_bar.png')


def print_summary(all_names, all_tr, all_vl):
    print('\n' + '=' * 60)
    print(f"{'Algorithm':<12}  {'Train Loss':>14}  {'Val Loss':>12}")
    print('=' * 60)
    for n, tr, vl in zip(all_names, all_tr, all_vl):
        print(f'{n:<12}  {tr[-1]:>14.6f}  {vl[-1]:>12.6f}')
    print('=' * 60)


# ══════════════════════════════════════════════════════════
#  SECTION 5 — MAIN
# ══════════════════════════════════════════════════════════

def main():
    EPOCHS = 200

    # ── Data ──────────────────────────────────────────────
    print('\n📦 Preparing dataset…')
    X_tr_b, X_te_b, y_tr_s, y_te_s, X_tr_raw, y_tr_raw, feat_names = make_dataset()
    _, D = X_tr_b.shape
    w0   = np.zeros(D)
    ep   = np.arange(1, EPOCHS + 1)

    plot_dataset(X_tr_raw, y_tr_raw, feat_names)

    # ── Train all optimizers ───────────────────────────────
    print('\n🚀 Training optimizers…')

    # 1. GD
    print(' → Gradient Descent')
    _, tr_gd, vl_gd, gn_gd = gradient_descent(
        X_tr_b, y_tr_s, X_te_b, y_te_s, w0, lr=0.05, epochs=EPOCHS)

    # 2. Mini-BGD (3 batch sizes)
    print(' → Mini-Batch GD')
    _, tr_bgd,      vl_bgd,  gn_bgd  = mini_batch_gd(X_tr_b, y_tr_s, X_te_b, y_te_s, w0, batch_size=256,  epochs=EPOCHS)
    _, tr_bgd32,    _,       _       = mini_batch_gd(X_tr_b, y_tr_s, X_te_b, y_te_s, w0, batch_size=32,   epochs=EPOCHS)
    _, tr_bgd1024,  _,       _       = mini_batch_gd(X_tr_b, y_tr_s, X_te_b, y_te_s, w0, batch_size=1024, epochs=EPOCHS)

    # 3. Momentum (3 betas)
    print(' → Momentum GD')
    _, tr_mom,   vl_mom,   gn_mom,   vel_mom  = momentum_gd(X_tr_b, y_tr_s, X_te_b, y_te_s, w0, beta=0.9,  epochs=EPOCHS)
    _, tr_momB5, _,        _,        _        = momentum_gd(X_tr_b, y_tr_s, X_te_b, y_te_s, w0, beta=0.5,  epochs=EPOCHS)
    _, tr_momB99,_,        _,        vel99    = momentum_gd(X_tr_b, y_tr_s, X_te_b, y_te_s, w0, beta=0.99, epochs=EPOCHS)

    # 4. AdaGrad (3 lrs)
    print(' → AdaGrad')
    _, tr_ada,    vl_ada,  gn_ada,   elr_ada  = adagrad(X_tr_b, y_tr_s, X_te_b, y_te_s, w0, lr=0.5,  epochs=EPOCHS)
    _, tr_ada01,  _,       _,        elr01    = adagrad(X_tr_b, y_tr_s, X_te_b, y_te_s, w0, lr=0.1,  epochs=EPOCHS)
    _, tr_ada10,  _,       _,        elr10    = adagrad(X_tr_b, y_tr_s, X_te_b, y_te_s, w0, lr=1.0,  epochs=EPOCHS)

    # 5. NAG
    print(' → NAG')
    _, tr_nag, vl_nag, gn_nag, la_nag = nag(X_tr_b, y_tr_s, X_te_b, y_te_s, w0, epochs=EPOCHS)

    # 6. AdaDelta (3 rhos)
    print(' → AdaDelta')
    _, tr_add,   vl_add,  gn_add,   un_add  = adadelta(X_tr_b, y_tr_s, X_te_b, y_te_s, w0, rho=0.95, epochs=EPOCHS)
    _, tr_add90, _,       _,        un90    = adadelta(X_tr_b, y_tr_s, X_te_b, y_te_s, w0, rho=0.90, epochs=EPOCHS)
    _, tr_add99, _,       _,        un99    = adadelta(X_tr_b, y_tr_s, X_te_b, y_te_s, w0, rho=0.99, epochs=EPOCHS)

    # 7. Adam (3 lrs)
    print(' → Adam')
    _, tr_adam,    vl_adam,   gn_adam, m1_adam, m2_adam = adam(X_tr_b, y_tr_s, X_te_b, y_te_s, w0, lr=0.01,  epochs=EPOCHS)
    _, tr_adam001, _,         _,       _,       _        = adam(X_tr_b, y_tr_s, X_te_b, y_te_s, w0, lr=0.001, epochs=EPOCHS)
    _, tr_adam1,   _,         _,       _,       _        = adam(X_tr_b, y_tr_s, X_te_b, y_te_s, w0, lr=0.1,   epochs=EPOCHS)

    # ── Individual Plots ───────────────────────────────────
    print('\n📊 Generating individual plots…')

    # 1. GD
    plot_3panel(
        title='1️⃣  Gradient Descent (Full-Batch GD)', fname='01_gradient_descent.png',
        color=COLORS['GD'], ep=ep, train_l=tr_gd, val_l=vl_gd,
        p3=gn_gd, p3_ylabel='||∇L||', p3_title='Gradient Norm per Epoch')

    # 2. BGD — batch-size comparison in panel 0, grad norm in panel 2
    plot_3panel(
        title='2️⃣  Mini-Batch Gradient Descent', fname='02_mini_batch_gd.png',
        color=COLORS['BGD'], ep=ep, train_l=tr_bgd, val_l=vl_bgd,
        p3=gn_bgd, p3_ylabel='Gradient Norm', p3_title='Avg Batch Gradient Norm (Noisy)',
        extra_p0=[
            (tr_bgd32,   '#ff6b6b', 'Batch=32'),
            (tr_bgd,     COLORS['BGD'], 'Batch=256'),
            (tr_bgd1024, '#4d96ff', 'Batch=1024'),
        ])

    # 3. Momentum — beta comparison in panel 0, velocity in panel 2
    plot_3panel(
        title='3️⃣  Momentum-Based Gradient Descent', fname='03_momentum_gd.png',
        color=COLORS['Momentum'], ep=ep, train_l=tr_mom, val_l=vl_mom,
        p3=None, p3_ylabel='||v||', p3_title='Velocity Norm Over Epochs',
        extra_p0=[
            (tr_momB5,  '#ff6b6b', 'β=0.5'),
            (tr_mom,    COLORS['Momentum'], 'β=0.9'),
            (tr_momB99, '#4d96ff', 'β=0.99'),
        ],
        extra_p3=[
            (vel_mom, COLORS['Momentum'], 'β=0.9'),
            (vel99,   '#4d96ff',          'β=0.99'),
        ])

    # 4. AdaGrad — lr comparison in panel 0, effective lr in panel 2
    plot_3panel(
        title='4️⃣  AdaGrad — Adaptive Gradient Algorithm', fname='04_adagrad.png',
        color=COLORS['AdaGrad'], ep=ep, train_l=tr_ada, val_l=vl_ada,
        p3=None, p3_ylabel='Avg Effective LR', p3_title='Effective LR Decay (Vanishing Problem!)',
        extra_p0=[
            (tr_ada01, '#ff6b6b',           'lr=0.1'),
            (tr_ada,   COLORS['AdaGrad'],   'lr=0.5'),
            (tr_ada10, '#6bcb77',           'lr=1.0'),
        ],
        extra_p3=[
            (elr01,   '#ff6b6b',         'lr=0.1'),
            (elr_ada, COLORS['AdaGrad'], 'lr=0.5'),
            (elr10,   '#6bcb77',         'lr=1.0'),
        ])

    # 5. NAG — compare with momentum in panel 0
    plot_3panel(
        title='5️⃣  NAG — Nesterov Accelerated Gradient', fname='05_nag.png',
        color=COLORS['NAG'], ep=ep, train_l=tr_nag, val_l=vl_nag,
        p3=la_nag, p3_ylabel='Lookahead Distance', p3_title='Lookahead Distance ||w - w̃||',
        extra_p0=[
            (tr_mom, COLORS['Momentum'], 'Momentum'),
            (tr_nag, COLORS['NAG'],      'NAG'),
        ])

    # 6. AdaDelta — rho comparison + update norms
    plot_3panel(
        title='6️⃣  AdaDelta — No Manual Learning Rate Required', fname='06_adadelta.png',
        color=COLORS['AdaDelta'], ep=ep, train_l=tr_add, val_l=vl_add,
        p3=None, p3_ylabel='||Δw||', p3_title='Update Step Norm ||Δw||',
        extra_p0=[
            (tr_add90, '#ff6b6b',           'ρ=0.90'),
            (tr_add,   COLORS['AdaDelta'],  'ρ=0.95'),
            (tr_add99, '#4d96ff',           'ρ=0.99'),
        ],
        extra_p3=[
            (un90,   '#ff6b6b',          'ρ=0.90'),
            (un_add, COLORS['AdaDelta'], 'ρ=0.95'),
            (un99,   '#4d96ff',          'ρ=0.99'),
        ])

    # 7. Adam — lr comparison + moment norms
    plot_3panel(
        title='7️⃣  Adam — Adaptive Moment Estimation', fname='07_adam.png',
        color=COLORS['Adam'], ep=ep, train_l=tr_adam, val_l=vl_adam,
        p3=None, p3_ylabel='Norm', p3_title='Bias-Corrected Moments m̂ & v̂',
        extra_p0=[
            (tr_adam001, '#ff6b6b',      'lr=0.001'),
            (tr_adam,    COLORS['Adam'], 'lr=0.01'),
            (tr_adam1,   '#6bcb77',      'lr=0.1'),
        ],
        extra_p3=[
            (m1_adam, '#ff9f1c', '1st Moment m̂'),
            (m2_adam, '#c77dff', '2nd Moment v̂'),
        ])

    # ── Comparison Plots ───────────────────────────────────
    print('\n📊 Generating comparison plots…')
    all_names = ['GD', 'BGD', 'Momentum', 'AdaGrad', 'NAG', 'AdaDelta', 'Adam']
    all_tr    = [tr_gd, tr_bgd, tr_mom, tr_ada, tr_nag, tr_add, tr_adam]
    all_vl    = [vl_gd, vl_bgd, vl_mom, vl_ada, vl_nag, vl_add, vl_adam]
    all_c     = [COLORS[n] for n in all_names]

    plot_all_comparison(ep, all_names, all_tr, all_vl, all_c)
    plot_bar_summary(all_names, all_tr, all_vl, all_c)

    # ── Console Summary ────────────────────────────────────
    print_summary(all_names, all_tr, all_vl)
    print('\n🎉 All done!  10 plots saved in the current directory.\n')


if __name__ == '__main__':
    main()


📦 Preparing dataset…
  ✅ Saved 00_dataset_overview.png

🚀 Training optimizers…
 → Gradient Descent
 → Mini-Batch GD
 → Momentum GD
 → AdaGrad
 → NAG
 → AdaDelta
 → Adam

📊 Generating individual plots…
  ✅ Saved 01_gradient_descent.png
  ✅ Saved 02_mini_batch_gd.png
  ✅ Saved 03_momentum_gd.png
  ✅ Saved 04_adagrad.png
  ✅ Saved 05_nag.png
  ✅ Saved 06_adadelta.png
  ✅ Saved 07_adam.png

📊 Generating comparison plots…
  ✅ Saved 08_all_comparison.png
  ✅ Saved 09_final_summary_bar.png

Algorithm         Train Loss      Val Loss
GD                  0.214191      0.214068
BGD                 0.214631      0.214453
Momentum            0.214191      0.214068
AdaGrad             0.214191      0.214068
NAG                 0.214191      0.214068
AdaDelta            0.233704      0.236901
Adam                0.214191      0.214064

🎉 All done!  10 plots saved in the current directory.

